# 画像生成アプリケーションの構築（Azure OpenAI）

LLMはテキスト生成だけではありません。テキストの説明から画像も生成できます。画像というモダリティは、MedTech、建築、観光、ゲーム開発、マーケティングなどさまざまな分野で有用です。このレッスンでは、Microsoft Foundry上の最新の<strong>GPT Image</strong>モデルを使って作業します。

## 学習目標

このレッスンの終わりには、以下ができるようになります：

- 画像生成とは何か、それがどこで役立つのかを説明できる。
- `gpt-image`モデルファミリーと、それが従来のDALL·Eモデルとどう異なるかを理解できる。
- 画像生成アプリケーションを構築し、画像を編集できる。

## 画像生成とは？

画像生成モデルはテキストプロンプトから画像を作成します。`gpt-image`のような最新モデルは、訓練中にテキストと画像の関係を学習し、ランダムノイズを反復処理してプロンプトに合った画像に変換します。

よく知られている画像モデルのファミリーは以下の通りです：

- **`gpt-image`（OpenAI）** — このレッスンで使う最新世代。テキストから画像生成と、マスクを使った画像編集（インペインティング）に対応。
- **Midjourney** — 独自のサービスやDiscordベースのワークフローを持つ人気のサードパーティモデル。

> 古いOpenAI画像モデルの<strong>DALL·E 2</strong>と<strong>DALL·E 3</strong>はレガシーです。DALL·E 3は新規展開には使用できず、`create_variation`などの機能はDALL·E 2のみに存在しました。新規アプリケーションには`gpt-image`モデルを使用してください。

Microsoft Foundry上では、最新で最も高性能な画像モデルは**`gpt-image-2`**で推奨されるデフォルトです。`gpt-image-1.5`および`gpt-image-1-mini`も一般提供されています。

> **重要：** `gpt-image`モデルは生成した画像をURLではなく<strong>base64</strong>（`b64_json`）として返します。コードはbase64文字列をデコードしてバイト配列に変換し保存します — 画像のダウンロード用URLはありません。


## 最初の画像生成アプリケーションの構築

では、画像生成アプリケーションを構築するには何が必要でしょうか？以下のライブラリが必要です：

- **python-dotenv**、これはコードから離れて秘密情報を<em>.env</em>ファイルに保持するために強く推奨されるライブラリです。
- **openai**、OpenAI APIと対話するために使用するライブラリです。
- **pillow**、Pythonで画像を扱うためのライブラリです。

まだの場合は、[Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) の指示に従ってMicrosoft Foundryリソースとモデルを作成してください。モデルは **gpt-image-2**（最新のAzure OpenAI画像モデル；DALL·Eはレガシー）を選択してください。

1. 次の内容でファイル<em>.env</em>を作成します：

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    この情報はMicrosoft Foundryポータルのリソースの「Deployments」セクションで確認できます。



1. 上記のライブラリを *requirements.txt* というファイルに次のようにまとめます:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 次に、仮想環境を作成し、ライブラリをインストールします:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windowsの場合、以下のコマンドを使用して仮想環境を作成およびアクティベートします:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. <em>app.py</em>というファイルに以下のコードを追加します:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenvをインポートします
    dotenv.load_dotenv()

    # Azure OpenAI（Microsoft Foundry）クライアントを設定します。
    # 画像モデルには新しいAPIバージョンが必要です — モデルに必要なバージョンはFoundryのドキュメントで確認してください。
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 画像生成APIを使って画像を作成します
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ここにプロンプトテキストを入力してください
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 例: "gpt-image-2"
        )
        # 保存する画像のディレクトリを設定します
        image_dir = os.path.join(os.curdir, 'images')

        # もしディレクトリが存在しなければ作成します
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 画像パスを初期化します（ファイルタイプはpngにしてください）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-imageモデルは画像をURLではなくbase64（b64_json）形式で返します
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 既定の画像ビューアで画像を表示します
        image = Image.open(image_path)
        image.show()

    # 例外をキャッチします
    except BadRequestError as err:
        print(err)

    ```

このコードを説明します:

- 最初に、OpenAIライブラリ、dotenvライブラリ、base64モジュール、およびPillowライブラリを含む必要なライブラリをインポートします。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 次に、<em>.env</em>ファイルから環境変数をロードします。

    ```python
    # dotenvをインポートする
    dotenv.load_dotenv()
    ```

- その後、Azure OpenAI（Microsoft Foundry）クライアントを設定します。

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 次に、画像を生成します:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # ここにあなたのプロンプトテキストを入力してください
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image`モデルは画像を`data[0].b64_json`で<strong>base64</strong>文字列として返します。これをバイトにデコードしてファイルに書き込みます — ダウンロード用のURLはありません。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後に、画像を開いて標準の画像ビューアで表示します:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 画像生成の詳細

`images.generate`のパラメータを見てみましょう:

- <strong>prompt</strong>は、画像生成に使用されるテキストプロンプトです。ここでは「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」となっています。
- <strong>size</strong>は生成される画像のサイズです（例: `1024x1024`、`1536x1024`、`1024x1536`、または`"auto"`）。
- <strong>n</strong>は生成する画像の数です。ここでは1つ生成します。
- <strong>model</strong>はあなたの画像モデルのデプロイ名です（例: `gpt-image-2`）。

> 画像モデルは`temperature`パラメータを取りません — それはテキスト生成用の制御です。多様性を得たい場合はAPIを再度呼び出し、多様性を減らしたい場合はプロンプトをより具体的にしてください。

## 画像生成の追加機能

Pythonの数行で画像を生成する方法を見ました。`gpt-image`モデルは、既存の画像を<strong>編集</strong>することもできます。画像、オプションの<strong>マスク</strong>（変更したい部分を示す）、およびプロンプトを提供することで、画像の一部を変更できます—例えば、ウサギに帽子を追加するなどです。

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編集内容もbase64で返されます
edited_image = base64.b64decode(response.data[0].b64_json)
```

元の画像にはウサギだけがあり、最終画像にはウサギに帽子が追加されています。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
